# HMM Momentum Trading — Pipeline Walkthrough

**Course:** Advanced Signal Processing: Tools and Applications (ASPTA), UPC Barcelona

**Paper:** Christensen, Turner & Godsill (2020), *"Hidden Markov Models Applied To Intraday Momentum Trading With Side Information"* (arXiv:2006.08307)

This notebook walks through the complete pipeline:
1. **Data loading & exploration** — daily prices, log-returns, descriptive statistics
2. **Model selection** — AIC/BIC to choose the number of hidden states K
3. **EM training** — Baum-Welch algorithm convergence
4. **Regime detection** — Viterbi decoding and state posteriors
5. **Online inference** — predict-update filtering loop
6. **Signal generation** — converting posteriors to trading signals
7. **Backtesting** — out-of-sample performance evaluation
8. **Signal refinement** — no-trade zone and EMA smoothing

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 100

## 1. Data Loading & Exploration

We use daily closing prices for SPY (S&P 500 ETF) from 2015 to 2024. Log-returns are computed as:

$$\Delta y_t = \log P_t - \log P_{t-1}$$

In [ ]:
from src.data.loader import load_daily_prices, extract_close_series
from src.data.features import log_returns

prices = load_daily_prices("SPY", "2015-01-01", "2024-12-31")
close = extract_close_series(prices)
returns = log_returns(close)

print(f"Observations: {len(returns)}")
print(f"Period: {returns.index.min().date()} to {returns.index.max().date()}")
print(f"\nDescriptive statistics:")
print(f"  Mean daily return:   {returns.mean():.6f}")
print(f"  Std daily return:    {returns.std():.6f}")
print(f"  Annualized return:   {returns.mean() * 252 * 100:.2f}%")
print(f"  Annualized vol:      {returns.std() * np.sqrt(252) * 100:.2f}%")
print(f"  Skewness:            {float(returns.skew()):.2f}")
print(f"  Excess kurtosis:     {float(returns.kurtosis()):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Price series
axes[0].plot(close.index, close.values, linewidth=0.8)
axes[0].set_title("SPY Daily Close Price")
axes[0].set_ylabel("Price ($)")
axes[0].grid(alpha=0.3)

# Return distribution with Gaussian overlay
axes[1].hist(returns.values, bins=80, density=True, alpha=0.7, label="Empirical")
x = np.linspace(returns.min(), returns.max(), 300)
gaussian = np.exp(-0.5 * ((x - returns.mean()) / returns.std())**2) / (returns.std() * np.sqrt(2 * np.pi))
axes[1].plot(x, gaussian, "r-", linewidth=2, label="Gaussian fit")
axes[1].set_title("Log-Return Distribution")
axes[1].set_xlabel("Daily log-return")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 2. Model Selection (AIC/BIC)

We fit HMMs with K=1 to K=6 and select the model minimizing information criteria:

$$\text{AIC} = -2 \log \mathcal{L} + 2p, \qquad \text{BIC} = -2 \log \mathcal{L} + p \log T$$

where $p = K^2 + 2K - 1$ is the number of free parameters and $T$ is the number of observations.

In [ ]:
from src.hmm.model_selection import compute_aic, compute_bic

# Pre-computed results from experiment 02 (full K=1..10 sweep takes ~30 min)
# These values are reproducible via: python experiments/02_model_selection.py
obs = returns.to_numpy()
K_range = range(1, 7)
pre_computed_ll = {1: 7292.5, 2: 8152.2, 3: 8379.2, 4: 8455.5, 5: 8462.8, 6: 8483.2}

aic_vals = [compute_aic(pre_computed_ll[k], k) for k in K_range]
bic_vals = [compute_bic(pre_computed_ll[k], k, len(obs)) for k in K_range]
lls = [pre_computed_ll[k] for k in K_range]

for k in K_range:
    print(f"K={k}: LL={pre_computed_ll[k]:.1f}, AIC={aic_vals[k-1]:.1f}, BIC={bic_vals[k-1]:.1f}")

print(f"\nBest K by AIC: {list(K_range)[np.argmin(aic_vals)]}")
print(f"Best K by BIC: {list(K_range)[np.argmin(bic_vals)]}")
print("\nWe use K=3 for interpretability (bearish / neutral / bullish).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ks = list(K_range)

axes[0].plot(ks, aic_vals, "o-", label="AIC")
axes[0].plot(ks, bic_vals, "s-", label="BIC")
axes[0].set_xlabel("Number of states K")
axes[0].set_ylabel("Information criterion")
axes[0].set_title("Model Selection")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(ks, lls, "o-", color="green")
axes[1].set_xlabel("Number of states K")
axes[1].set_ylabel("Log-likelihood")
axes[1].set_title("Log-Likelihood vs K")
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 3. EM Training (Baum-Welch)

We train a K=3 HMM using Baum-Welch (EM) on the **training set** (70% of data). The algorithm iterates:

**E-step:** Compute forward variables $\alpha_t(k)$ and backward variables $\beta_t(k)$, then posterior responsibilities:
$$\gamma_t(k) = \frac{\alpha_t(k)\,\beta_t(k)}{\sum_j \alpha_t(j)\,\beta_t(j)}$$

**M-step:** Update parameters:
$$\hat{\mu}_k = \frac{\sum_t \gamma_t(k)\,\Delta y_t}{\sum_t \gamma_t(k)}, \qquad \hat{\sigma}^2_k = \frac{\sum_t \gamma_t(k)\,(\Delta y_t - \hat{\mu}_k)^2}{\sum_t \gamma_t(k)}$$

In [ ]:
from src.hmm.utils import sort_states, train_best_model

K = 3
split = int(len(returns) * 0.7)
train_returns = returns.iloc[:split]
test_returns = returns.iloc[split:]

print(f"Train: {train_returns.index.min().date()} to {train_returns.index.max().date()} ({len(train_returns)} obs)")
print(f"Test:  {test_returns.index.min().date()} to {test_returns.index.max().date()} ({len(test_returns)} obs)")

params, history, _ = train_best_model(
    train_returns.to_numpy(), K,
    successful_restarts=2, max_attempts=30,
    max_iter=100, tol=1e-6, random_state=42,
)
params = sort_states(params)

A, pi, mu, sigma2 = params["A"], params["pi"], params["mu"], params["sigma2"]

print(f"\nConverged in {len(history)} iterations")
print(f"Final log-likelihood: {history[-1]:.2f}")

labels = ["Bearish", "Neutral", "Bullish"]
print(f"\n{'State':<10}{'Label':<10}{'Daily mu':>12}{'Ann. Return':>14}{'Ann. Vol':>12}")
print("-" * 58)
for i in range(K):
    ann_ret = mu[i] * 252 * 100
    ann_vol = np.sqrt(sigma2[i]) * np.sqrt(252) * 100
    print(f"{i:<10}{labels[i]:<10}{mu[i]:>12.6f}{ann_ret:>13.2f}%{ann_vol:>11.2f}%")

print(f"\nTransition matrix A:")
print(np.array2string(A, precision=4, suppress_small=True))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(history) + 1), history, "o-", markersize=3)
ax.set_xlabel("EM iteration")
ax.set_ylabel("Log-likelihood")
ax.set_title(f"Baum-Welch Convergence (K={K})")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Regime Detection (Viterbi Decoding)

The Viterbi algorithm finds the most likely state sequence:
$$m_{1:T}^* = \arg\max_{m_{1:T}} p(m_{1:T} \mid \Delta y_{1:T}, \Theta)$$

using the recursion (in log-space):
$$\delta_t(k) = \max_i \left[\delta_{t-1}(i) + \log A_{ik}\right] + \log \mathcal{N}(\Delta y_t; \mu_k, \sigma^2_k)$$

In [ ]:
from src.hmm.viterbi import viterbi
from src.utils.plotting import plot_regime_colored_prices

# Apply Viterbi to full dataset using trained parameters
full_obs = returns.to_numpy()
states_full, _ = viterbi(full_obs, A, pi, mu, sigma2)

# close has T+1 prices, returns/states have T — align by dropping first price
fig, ax = plot_regime_colored_prices(close.values[1:], states_full)
ax.set_title("SPY Price Colored by Viterbi-Decoded Regime")
ax.set_xlabel("Trading day")
fig.tight_layout()
plt.show()

print("State counts:", {labels[i]: int((states_full == i).sum()) for i in range(K)})

## 5. Online Inference (Predict-Update Filter)

For trading, we use causal (online) filtering rather than batch smoothing. At each time step $t$, we:

**Predict:** $p(m_t = k \mid \Delta y_{1:t-1}) = \sum_i p(m_{t-1} = i \mid \Delta y_{1:t-1}) \cdot A_{ik}$

**Update:** $p(m_t = k \mid \Delta y_{1:t}) \propto p(m_t = k \mid \Delta y_{1:t-1}) \cdot \mathcal{N}(\Delta y_t; \mu_k, \sigma^2_k)$

This gives us the filtered state probabilities $\omega_{t,k}$ used to generate trading signals (Paper §6, Algorithm 4).

In [ ]:
from src.hmm.inference import run_inference

# Run online inference on test data using train-set parameters
predictions, state_probs = run_inference(test_returns.to_numpy(), A, pi, mu, sigma2)

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for k in range(K):
    axes[k].plot(test_returns.index, state_probs[:, k], linewidth=0.8)
    axes[k].set_ylabel(f"P({labels[k]})")
    axes[k].set_ylim(-0.05, 1.05)
    axes[k].grid(alpha=0.3)

axes[0].set_title("Filtered State Posteriors (Test Period)")
axes[-1].set_xlabel("Date")
fig.tight_layout()
plt.show()

## 6. Signal Generation

We convert filtered posteriors to trading signals using the **weighted vote**:

$$s_t = \sum_k \omega_{t,k} \cdot \text{sign}(\mu_k)$$

Each state votes long (+1) or short (-1) according to its emission mean, weighted by its posterior probability. The result is a continuous signal in [-1, 1].

In [ ]:
from src.strategy.signals import states_to_signal, predictions_to_signal

signals_vote = states_to_signal(state_probs, mu)
signals_sign = predictions_to_signal(predictions, transfer_fn="sign")

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)

axes[0].plot(test_returns.index, signals_sign, linewidth=0.7, alpha=0.8)
axes[0].set_ylabel("Sign signal")
axes[0].set_ylim(-1.3, 1.3)
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].grid(alpha=0.3)

axes[1].plot(test_returns.index, signals_vote, linewidth=0.7, alpha=0.8, color="tab:orange")
axes[1].set_ylabel("Weighted vote")
axes[1].set_ylim(-1.3, 1.3)
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].set_xlabel("Date")
axes[1].grid(alpha=0.3)

axes[0].set_title("Trading Signals (Test Period)")
fig.tight_layout()
plt.show()

print(f"Vote signal — mean: {signals_vote.mean():.3f}, "
      f"frac long: {(signals_vote > 0).mean():.1%}, "
      f"frac short: {(signals_vote < 0).mean():.1%}")

## 7. Backtesting

We simulate out-of-sample trading with a one-period execution lag and 5 bps transaction costs:

$$r_t^{\text{strat}} = s_{t-1} \cdot \Delta y_t, \qquad r_t^{\text{net}} = r_t^{\text{strat}} - |s_{t-1} - s_{t-2}| \cdot c$$

where $s_0 = 0$ (start flat) and $c = 5 / 10000$.

In [ ]:
from src.strategy.backtest import backtest

test_np = test_returns.to_numpy()
COST = 5

result_vote = backtest(test_np, signals_vote, transaction_cost_bps=COST)
result_sign = backtest(test_np, signals_sign, transaction_cost_bps=COST)
result_bh = backtest(test_np, np.ones_like(test_np), transaction_cost_bps=0)

print(f"{'Strategy':<20}{'Sharpe':>8}{'Ann.Return':>12}{'MaxDrawdown':>13}{'Turnover':>10}")
print("-" * 63)
for name, r in [("Weighted vote", result_vote), ("Sign signal", result_sign),
                ("Buy-and-hold", result_bh)]:
    m = r["metrics"]
    print(f"{name:<20}{m['sharpe']:>8.2f}{m['annualized_return']*100:>11.2f}%"
          f"{m['max_drawdown']*100:>12.2f}%{m['turnover']:>10.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(test_returns.index, result_vote["cumulative"], label="Weighted vote", linewidth=1.8)
ax.plot(test_returns.index, result_sign["cumulative"], label="Sign signal", linewidth=1.8)
ax.plot(test_returns.index, result_bh["cumulative"], label="Buy-and-hold",
        linewidth=1.8, linestyle="--")
ax.set_title("Out-of-Sample Backtest: Cumulative Returns")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative value")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 8. Signal Refinement

Two post-processing techniques improve risk-adjusted performance:

**No-trade zone:** Zero the signal when the neutral-state posterior exceeds a threshold $\tau$:
$$s'_t = s_t \cdot \mathbb{1}(\omega_{t,\text{neutral}} < \tau)$$

**EMA smoothing:** Dampen rapid signal changes:
$$s''_t = \alpha \cdot s'_t + (1 - \alpha) \cdot s''_{t-1}$$

The best configuration from grid search is $\tau = 0.6$, $\alpha = 0.1$.

In [ ]:
from src.strategy.signals import apply_no_trade_zone, smooth_signal

# Apply best refinement config from experiment 10
THRESHOLD = 0.6
ALPHA = 0.1
NEUTRAL_IDX = 1

signals_refined = states_to_signal(state_probs, mu)
signals_refined = apply_no_trade_zone(signals_refined, state_probs, NEUTRAL_IDX, THRESHOLD)
signals_refined = smooth_signal(signals_refined, ALPHA)

result_refined = backtest(test_np, signals_refined, transaction_cost_bps=COST)

print(f"{'Strategy':<25}{'Sharpe':>8}{'Ann.Return':>12}{'MaxDrawdown':>13}{'Turnover':>10}")
print("-" * 68)
for name, r in [("Baseline (vote)", result_vote),
                ("Refined (thr=0.6, a=0.1)", result_refined),
                ("Buy-and-hold", result_bh)]:
    m = r["metrics"]
    print(f"{name:<25}{m['sharpe']:>8.2f}{m['annualized_return']*100:>11.2f}%"
          f"{m['max_drawdown']*100:>12.2f}%{m['turnover']:>10.4f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))

# Cumulative returns comparison
axes[0].plot(test_returns.index, result_vote["cumulative"], label="Baseline (vote)", linewidth=1.8)
axes[0].plot(test_returns.index, result_refined["cumulative"],
             label=f"Refined (thr={THRESHOLD}, α={ALPHA})", linewidth=1.8)
axes[0].plot(test_returns.index, result_bh["cumulative"], label="Buy-and-hold",
             linewidth=1.8, linestyle="--")
axes[0].set_title("Signal Refinement: Cumulative Returns")
axes[0].set_ylabel("Cumulative value")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Signal comparison
axes[1].plot(test_returns.index, signals_vote, alpha=0.5, linewidth=0.7, label="Baseline")
axes[1].plot(test_returns.index, signals_refined, linewidth=1.0, label="Refined")
axes[1].set_ylabel("Signal")
axes[1].set_xlabel("Date")
axes[1].set_ylim(-1.3, 1.3)
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## Summary

| Step | Method | Key Result |
|------|--------|------------|
| Model selection | AIC/BIC | K=4 optimal, K=3 used for interpretability |
| Training | Baum-Welch EM | 3 regimes: bearish (56% vol), neutral (20% vol), bullish (8.5% vol) |
| Regime detection | Viterbi | Bearish episodes align with COVID-19 and 2022 drawdown |
| Backtesting | Weighted vote | Sharpe 0.54 vs 0.40 buy-and-hold, max drawdown 20% vs 27% |
| Signal refinement | NTZ + EMA | Sharpe 0.71, max drawdown 7.8% |

The HMM successfully identifies latent momentum regimes and generates trading signals that improve risk-adjusted returns relative to buy-and-hold, with the signal refinement step providing substantial further improvement in drawdown control.